# 03 - Gold Dimensional Model

## Purpose

Create a business-ready Gold star schema for Retail Banking Customer Analytics.

Dimensions: dim_customer, dim_account, dim_product, dim_branch, dim_date.

Fact: fact_transaction.

The Gold layer uses surrogate keys to simplify Power BI relationships and preserve stable reporting joins. Natural keys are retained for traceability.

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window


def add_surrogate_key(df, key_name, order_columns):
    window_spec = Window.orderBy(*[F.col(c) for c in order_columns])
    return df.withColumn(key_name, F.row_number().over(window_spec).cast("long"))


def write_gold(df, table_name):
    df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(table_name)
    print(f"{table_name}: {df.count()} rows written")

In [ ]:
customers = spark.table("silver_customers")
accounts = spark.table("silver_accounts")
products = spark.table("silver_products")
branches = spark.table("silver_branches")
transactions = spark.table("silver_transactions")

In [ ]:
dim_customer = customers.select("customer_id", "first_name", "last_name", "email", "phone", "city", "state", "country", "customer_segment", "customer_status", "date_of_birth", "created_date").withColumn("customer_full_name", F.concat_ws(" ", "first_name", "last_name")).withColumn("is_active_customer", F.when(F.col("customer_status") == "Active", F.lit(True)).otherwise(F.lit(False)))
dim_customer = add_surrogate_key(dim_customer, "customer_key", ["customer_id"])
write_gold(dim_customer, "dim_customer")

In [ ]:
dim_account = accounts.select("account_id", "customer_id", "product_id", "branch_id", "account_type", "account_status", "opened_date", "closed_date", "current_balance").withColumn("is_open_account", F.when(F.col("account_status") == "Open", F.lit(True)).otherwise(F.lit(False)))
dim_account = add_surrogate_key(dim_account, "account_key", ["account_id"])
write_gold(dim_account, "dim_account")

In [ ]:
dim_product = products.select("product_id", "product_name", "product_category", "product_family", "fee_model", "is_active", "launch_date")
dim_product = add_surrogate_key(dim_product, "product_key", ["product_id"])
write_gold(dim_product, "dim_product")

dim_branch = branches.select("branch_id", "branch_name", "city", "state", "region", "opened_date", "is_active")
dim_branch = add_surrogate_key(dim_branch, "branch_key", ["branch_id"])
write_gold(dim_branch, "dim_branch")

In [ ]:
min_max_dates = transactions.agg(F.min("transaction_date").alias("min_date"), F.max("transaction_date").alias("max_date")).collect()[0]
min_date = min_max_dates["min_date"]
max_date = min_max_dates["max_date"]

dim_date = (
    spark.sql(f"SELECT explode(sequence(to_date('{min_date}'), to_date('{max_date}'), interval 1 day)) AS date_value")
    .withColumn("date_key", F.date_format("date_value", "yyyyMMdd").cast("int"))
    .withColumn("year", F.year("date_value"))
    .withColumn("quarter", F.quarter("date_value"))
    .withColumn("month", F.month("date_value"))
    .withColumn("month_name", F.date_format("date_value", "MMMM"))
    .withColumn("year_month", F.date_format("date_value", "yyyy-MM"))
    .withColumn("day_of_month", F.dayofmonth("date_value"))
    .withColumn("day_name", F.date_format("date_value", "EEEE"))
    .withColumn("is_weekend", F.dayofweek("date_value").isin([1, 7]))
)
write_gold(dim_date, "dim_date")

In [ ]:
fact_transaction = (
    transactions.alias("t")
    .join(dim_account.select("account_key", "account_id", "customer_id"), "account_id", "left")
    .join(dim_customer.select("customer_key", "customer_id"), "customer_id", "left")
    .join(dim_product.select("product_key", "product_id"), "product_id", "left")
    .join(dim_branch.select("branch_key", "branch_id"), "branch_id", "left")
    .withColumn("date_key", F.date_format("transaction_date", "yyyyMMdd").cast("int"))
    .select("transaction_id", "date_key", "customer_key", "account_key", "product_key", "branch_key", "transaction_timestamp", "transaction_date", "transaction_type", "transaction_channel", "transaction_status", "currency", "transaction_amount", "_batch_id")
    .withColumn("is_posted", F.when(F.col("transaction_status") == "Posted", F.lit(True)).otherwise(F.lit(False)))
    .withColumn("absolute_transaction_amount", F.abs(F.col("transaction_amount")))
)
write_gold(fact_transaction, "fact_transaction")

In [ ]:
validation = fact_transaction.select(
    F.count("*").alias("fact_rows"),
    F.sum(F.when(F.col("customer_key").isNull(), 1).otherwise(0)).alias("missing_customer_key"),
    F.sum(F.when(F.col("account_key").isNull(), 1).otherwise(0)).alias("missing_account_key"),
    F.sum(F.when(F.col("product_key").isNull(), 1).otherwise(0)).alias("missing_product_key"),
    F.sum(F.when(F.col("branch_key").isNull(), 1).otherwise(0)).alias("missing_branch_key")
)
display(validation)
row = validation.collect()[0].asDict()
if any(row[k] > 0 for k in row if k.startswith("missing_")):
    raise ValueError("Gold validation failed because one or more fact rows have missing dimension keys")

In [ ]:
sample = (
    spark.table("fact_transaction").alias("f")
    .join(spark.table("dim_date").alias("d"), "date_key")
    .join(spark.table("dim_product").alias("p"), "product_key")
    .filter(F.col("f.is_posted") == True)
    .groupBy("d.year_month", "p.product_category")
    .agg(F.count("*").alias("transaction_count"), F.sum("absolute_transaction_amount").alias("total_transaction_amount"))
    .orderBy("d.year_month", "p.product_category")
)
display(sample)